# Amazon ML Challenge 2026 — Phase 2A

Thin Kaggle launcher for target→S1 exact and structured retrieval. Attach both the raw competition dataset and the saved Phase-1 normalization output, enable Internet, select a CPU session, and run all cells. Production logic stays in the pinned repository commit.

Resource note: this is CPU/I/O work; a GPU is not required. **UNVERIFIED estimate:** roughly 30–120 minutes depending on Kaggle storage and CPU throughput.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/yugtheguy/NEW_AMAZON.git'
PINNED_COMMIT = '138d7e6933d40f4a3f616307143d2431968d3984'
INPUT_ROOT = Path('/kaggle/input')
REPO_DIR = Path('/kaggle/working/NEW_AMAZON')
RESUME_ARTIFACTS_ROOT = None  # Optional Path to a prior saved artifacts directory
print({'repo': REPO_URL, 'commit': PINNED_COMMIT})

In [ ]:
expected_raw = {
    'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv',
    'train_ground_truth.tsv', 'test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv',
}
raw_matches = {name: list(INPUT_ROOT.rglob(name)) for name in sorted(expected_raw)}
problems = {name: paths for name, paths in raw_matches.items() if len(paths) != 1}
if problems:
    raise RuntimeError(f'Each raw file must occur exactly once under {INPUT_ROOT}: {problems}')
manifest_paths = list(INPUT_ROOT.rglob('part-*.manifest.json'))
normalized_roots = sorted({
    path.parents[3] for path in manifest_paths
    if len(path.parents) > 3 and path.parents[3].name == 'v1'
    and path.parents[3].parent.name == 'normalized'
})
if len(normalized_roots) != 1:
    raise RuntimeError(f'Expected exactly one normalized/v1 input root, found: {normalized_roots}')
NORMALIZED_ROOT = normalized_roots[0]
print('Normalized input:', NORMALIZED_ROOT)
print('Normalized manifests:', len(list(NORMALIZED_ROOT.rglob('part-*.manifest.json'))))

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
else:
    status = subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout
    if status.strip():
        raise RuntimeError(f'Existing repository is not clean: {status}')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', PINNED_COMMIT], check=True)
actual = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert actual == PINNED_COMMIT, (actual, PINNED_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[data]'], check=True)
print('Checked out and installed', actual)

In [ ]:
artifact_root = REPO_DIR / 'artifacts'
if RESUME_ARTIFACTS_ROOT is not None:
    resume_root = Path(RESUME_ARTIFACTS_ROOT)
    if not resume_root.exists():
        raise FileNotFoundError(resume_root)
    for relative in (Path('retrieval/v1/exact_structured'), Path('reports/retrieval/v1/exact_structured')):
        source = resume_root / relative
        destination = artifact_root / relative
        if source.exists():
            shutil.copytree(source, destination, dirs_exist_ok=True)
            print('Restored', source, '->', destination)
else:
    print('Fresh run: no Phase 2A checkpoint configured')
subprocess.run(['bash', '-lc', 'free -h && df -h /kaggle/working'], check=True)

In [ ]:
log_path = Path('/kaggle/working/phase2a_exact_structured.log')
command = [
    sys.executable, '-m', 'amazon_er.cli', 'run',
    '--stage', 'retrieve-exact-structured',
    '--config', str(REPO_DIR / 'configs/prod.yaml'),
    '--normalized-root', str(NORMALIZED_ROOT),
    '--data-root', str(INPUT_ROOT),
]
print('Running:', ' '.join(command))
with log_path.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Phase 2A failed with exit code {return_code}; inspect {log_path}')
print('Phase 2A completed successfully')

In [ ]:
summary_path = REPO_DIR / 'artifacts/reports/retrieval/v1/exact_structured/summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
assert summary['status'] == 'complete'
assert summary['stage'] == 'exact_structured'
assert summary['train_recall'] is not None
parquets = sorted((REPO_DIR / 'artifacts/retrieval/v1/exact_structured').rglob('part-*.parquet'))
manifests = sorted((REPO_DIR / 'artifacts/retrieval/v1/exact_structured').rglob('part-*.manifest.json'))
assert parquets and len(parquets) == len(manifests), (len(parquets), len(manifests))
print(json.dumps(summary, indent=2, sort_keys=True))
print({'candidate_parquets': len(parquets), 'manifests': len(manifests), 'log': str(log_path)})
print('Use Save Version to preserve /kaggle/working/NEW_AMAZON/artifacts and the log.')